# 01 - Generate Moveset

Este notebook processa um video de danca com MediaPipe Pose Landmarker (Tasks API), gera um video anotado com o esqueleto detectado e salva um moveset em JSON para comparacoes futuras em tempo real.

In [1]:
VIDEO_PATH = "caminho/do/video.mp4"

In [3]:
#!pip install mediapipe

   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/10.9 MB 10.3 MB/s eta 0:00:01
   -------------- ------------------------- 3.9/10.9 MB 11.2 MB/s eta 0:00:01
   ------------------------ --------------- 6.8/10.9 MB 12.6 MB/s eta 0:00:01
   ---------------------------------------  10.7/10.9 MB 14.3 MB/s eta 0:00:01
   ---------------------------------------- 10.9/10.9 MB 14.2 MB/s  0:00:00
   ---------------------------------------- 0.0/53.8 MB ? eta -:--:--
   -- ------------------------------------- 3.7/53.8 MB 19.4 MB/s eta 0:00:03
   ----- ---------------------------------- 7.9/53.8 MB 21.6 MB/s eta 0:00:03
   --------- ------------------------------ 13.1/53.8 MB 22.5 MB/s eta 0:00:02
   ------------- -------------------------- 18.1/53.8 MB 23.0 MB/s eta 0:00:02
   ---------------- ----------------------- 22.8/53.8 MB 23.2 MB/s eta 0:00:02
   --------------------- ------------------ 28.6/53.8 MB 24.0 MB/s eta 0:00:02


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\dvmrn\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [4]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
class MoveSetGenerator:
    """Generate an annotated pose video and a reusable temporal moveset."""

    LANDMARK_COUNT = 33
    POSE_CONNECTIONS: tuple[tuple[int, int], ...] = (
        (0, 1), (1, 2), (2, 3), (3, 7),
        (0, 4), (4, 5), (5, 6), (6, 8),
        (9, 10),
        (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
        (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
        (11, 23), (12, 24), (23, 24),
        (23, 25), (24, 26), (25, 27), (26, 28),
        (27, 29), (28, 30), (29, 31), (30, 32), (27, 31), (28, 32),
    )

    def __init__(self) -> None:
        self.video_path: Path | None = None
        self.model_path: Path | None = None
        self.annotated_video_path: Path | None = None
        self.moveset_path: Path | None = None
        self.capture: cv2.VideoCapture | None = None
        self.writer: cv2.VideoWriter | None = None
        self.detector: vision.PoseLandmarker | None = None
        self.fps = 0.0
        self.frame_count = 0
        self.width = 0
        self.height = 0
        self.duration = 0.0
        self.frames: list[dict[str, Any]] = []
        self.detected_frames = 0
        self.missing_pose_frames = 0

    def initialize_detector(self) -> None:
        """Load a local MediaPipe Pose Landmarker .task model."""
        self.model_path = self._find_task_model()
        base_options = python.BaseOptions(model_asset_path=str(self.model_path))
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            num_poses=1,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5,
            output_segmentation_masks=False,
        )
        self.detector = vision.PoseLandmarker.create_from_options(options)

    def load_video(self, video_path: str | Path) -> None:
        """Open the source video and extract its metadata."""
        self.video_path = Path(video_path).expanduser().resolve()
        if not self.video_path.exists():
            raise FileNotFoundError(f"Video nao encontrado: {self.video_path}")

        self.capture = cv2.VideoCapture(str(self.video_path))
        if not self.capture.isOpened():
            raise RuntimeError(f"Nao foi possivel abrir o video: {self.video_path}")

        self.fps = float(self.capture.get(cv2.CAP_PROP_FPS))
        self.frame_count = int(self.capture.get(cv2.CAP_PROP_FRAME_COUNT))
        self.width = int(self.capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

        if self.fps <= 0 or self.frame_count <= 0 or self.width <= 0 or self.height <= 0:
            raise ValueError("Metadados invalidos no video de entrada.")

        self.duration = self.frame_count / self.fps
        self.annotated_video_path = self.video_path.with_name(f"{self.video_path.stem}_pose.mp4")
        self.moveset_path = self.video_path.with_name(f"{self.video_path.stem}_moveset.json")

    def process_video(self, video_path: str | Path) -> None:
        """Process every video frame and write all requested outputs."""
        self.load_video(video_path)
        self.initialize_detector()
        self._initialize_writer()

        try:
            assert self.capture is not None
            for frame_index in range(self.frame_count):
                success, frame = self.capture.read()
                if not success:
                    break

                annotated_frame = self.process_frame(frame, frame_index)
                self.save_annotated_video(annotated_frame)
        finally:
            self._release_resources()

        self.save_json()
        self.print_summary()

    def process_frame(self, frame: np.ndarray, frame_index: int) -> np.ndarray:
        """Run pose detection for one frame and store its moveset data."""
        if self.detector is None:
            raise RuntimeError("Detector nao inicializado.")

        timestamp_ms = int(round(frame_index * 1000.0 / self.fps))
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        result = self.detector.detect_for_video(mp_image, timestamp_ms)
        pose_landmarks = result.pose_landmarks[0] if result.pose_landmarks else None

        if pose_landmarks:
            landmarks = [
                [round(point.x, 6), round(point.y, 6), round(point.z, 6), round(point.visibility, 6)]
                for point in pose_landmarks
            ]
            self.detected_frames += 1
            annotated_frame = self.draw_pose(frame.copy(), pose_landmarks)
        else:
            landmarks = []
            self.missing_pose_frames += 1
            annotated_frame = frame

        self.frames.append(
            {
                "frame": frame_index,
                "timestamp": round(timestamp_ms / 1000.0, 3),
                "pose_detected": bool(pose_landmarks),
                "landmarks": landmarks,
            }
        )
        return annotated_frame

    def draw_pose(self, frame: np.ndarray, landmarks: list[Any]) -> np.ndarray:
        """Draw the official pose skeleton connections over a frame."""
        points = []
        for landmark in landmarks:
            x = int(round(landmark.x * self.width))
            y = int(round(landmark.y * self.height))
            points.append((x, y, landmark.visibility))

        for start_idx, end_idx in self.POSE_CONNECTIONS:
            start = points[start_idx]
            end = points[end_idx]
            if start[2] >= 0.35 and end[2] >= 0.35:
                cv2.line(frame, start[:2], end[:2], (0, 255, 0), 2, cv2.LINE_AA)

        for x, y, visibility in points:
            if visibility >= 0.35:
                cv2.circle(frame, (x, y), 4, (0, 128, 255), -1, cv2.LINE_AA)

        return frame

    def save_json(self) -> None:
        """Save the moveset JSON optimized for fast game reads."""
        if self.video_path is None or self.moveset_path is None:
            raise RuntimeError("Caminhos de saida nao inicializados.")

        moveset = {
            "metadata": {
                "video": self.video_path.name,
                "fps": self.fps,
                "frame_count": self.frame_count,
                "width": self.width,
                "height": self.height,
                "duration": round(self.duration, 3),
                "landmark_count": self.LANDMARK_COUNT,
            },
            "frames": self.frames,
        }

        with self.moveset_path.open("w", encoding="utf-8") as file:
            json.dump(moveset, file, ensure_ascii=False, separators=(",", ":"))

    def save_annotated_video(self, frame: np.ndarray) -> None:
        """Append one annotated frame to the output video."""
        if self.writer is None:
            raise RuntimeError("VideoWriter nao inicializado.")
        self.writer.write(frame)

    def print_summary(self) -> None:
        """Print processing metadata and output locations."""
        print("Resumo do processamento")
        print(f"Video original: {self.video_path}")
        print(f"Video anotado: {self.annotated_video_path}")
        print(f"JSON gerado: {self.moveset_path}")
        print(f"Resolucao: {self.width}x{self.height}")
        print(f"FPS: {self.fps:.3f}")
        print(f"Duracao: {self.duration:.3f}s")
        print(f"Quantidade de frames: {self.frame_count}")
        print(f"Frames com pose detectada: {self.detected_frames}")
        print(f"Frames sem pose detectada: {self.missing_pose_frames}")

    def _find_task_model(self) -> Path:
        """Return the exact Pose Landmarker model path requested by the user."""
        model_path = Path(r"C:/Users/dvmrn/Repositórios/ApenasDance-AiMotionTrackingDanceGame/pose_landmarker_full.task")
        if not model_path.exists():
            raise FileNotFoundError(f"Modelo .task nao encontrado: {model_path}")
        return model_path.resolve()

    def _initialize_writer(self) -> None:
        """Create the MP4 writer using the original video properties."""
        if self.annotated_video_path is None:
            raise RuntimeError("Caminho do video anotado nao inicializado.")

        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        self.writer = cv2.VideoWriter(
            str(self.annotated_video_path),
            fourcc,
            self.fps,
            (self.width, self.height),
        )
        if not self.writer.isOpened():
            raise RuntimeError(f"Nao foi possivel criar o video: {self.annotated_video_path}")

    def _release_resources(self) -> None:
        """Release native resources even when processing fails."""
        if self.capture is not None:
            self.capture.release()
        if self.writer is not None:
            self.writer.release()
        if self.detector is not None:
            self.detector.close()

In [ ]:
generator = MoveSetGenerator()
generator.process_video("C:/Users/dvmrn/Repositórios/ApenasDance-AiMotionTrackingDanceGame/dance2.mp4")

RuntimeError: Nao foi possivel abrir o video: C:\Users\dvmrn\Repositórios\ApenasDance-AiMotionTrackingDanceGame